# Claude API — Features Deep Dive

**Live online course — instructor walkthrough notebook**

This notebook follows the lecture notes on advanced Claude features. Each section has:
- **Concept markdown** — what to explain on screen.
- **Demo code** — small, runnable cells that visibly demonstrate the concept.
- **🏫 During class** callouts — specific instructor actions, talking points, and variations.

---

## Agenda

1. Extended Thinking
2. Image Support
3. PDF Support
4. Citations
5. Prompt Caching — concept
6. Rules of Prompt Caching
7. Prompt Caching in Action
8. Code Execution and the Files API
9. Recap + practice exercises

## 0. Setup (do this before class starts)

1. Install dependencies:
   ```bash
   pip install anthropic python-dotenv
   ```
2. Create a file named `.env` next to this notebook containing:
   ```
   ANTHROPIC_API_KEY="sk-ant-...your-key..."
   ```
3. Add `.env` to `.gitignore` so the key is never committed.

> **🏫 During class:** Open `.env` (blur the key) and emphasize: **never paste API keys into a notebook cell** — `.env` + `python-dotenv` keeps secrets out of source control. The next code cell prints `Key loaded: True/False` so a missing `.env` is caught before any API call.

In [ ]:
# Install packages (uncomment if not already installed)
# %pip install anthropic python-dotenv

The setup cell loads the `.env` file, creates the SDK client, and pins a default `model` constant. The three `print()` lines at the end act as a sanity check: SDK version, model name, and `Key loaded: True/False`.

We pass two beta headers to the client up front: one enables Claude's **Code Execution** tool (used in §8), the other enables the **Files API** for uploading data the tool can read. They're harmless for every other demo in the notebook, so we set them once here and forget about it.

In [ ]:
from dotenv import load_dotenv
import anthropic
import os

load_dotenv()

client = anthropic.Anthropic(
    default_headers={
        "anthropic-beta": "code-execution-2025-08-25, files-api-2025-04-14"
    }
)

model = "claude-sonnet-4-6"

print("SDK version:", anthropic.__version__)
print("Model:", model)
print("Key loaded:", bool(os.getenv("ANTHROPIC_API_KEY")))

A trio of helpers we'll reuse for the rest of the notebook:

- `add_user_message` / `add_assistant_message` — append a turn to a `messages` list. We're keeping the conversation history ourselves; the API stores nothing between calls.
- `chat(...)` — send the messages list to Claude and return the **full Message object** (not just the text). The advanced features in this notebook need access to thinking blocks, citation arrays, cache-token counts, and tool-result blocks — all of which live on the response object.
- `text_from_message(...)` — pull just the text out when that's all we want.

The `chat()` helper accepts optional `system`, `tools`, `thinking`, and `thinking_budget` arguments — every demo below varies one or two of these and reuses the same function.

In [ ]:
def add_user_message(messages, content):
    messages.append({"role": "user", "content": content})
    return messages

def add_assistant_message(messages, content):
    messages.append({"role": "assistant", "content": content})
    return messages

def chat(messages, system=None, temperature=1.0, tools=None,
         thinking=False, thinking_budget=2000, max_tokens=4000):
    """Send messages to Claude and return the full Message object."""
    params = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": messages,
        "temperature": temperature,
    }
    if system is not None:
        params["system"] = system
    if tools is not None:
        params["tools"] = tools
    if thinking:
        params["thinking"] = {"type": "enabled", "budget_tokens": thinking_budget}
    return client.messages.create(**params)

def text_from_message(message):
    return "\n".join(b.text for b in message.content if b.type == "text")

---
# 1. Extended Thinking

**Extended Thinking** lets Claude reason for a budget of "thinking tokens" before producing its final response. The reasoning is visible to you (and your users), so you can audit how the model arrived at an answer.

### Mechanics
- **Thinking budget** — minimum **1024** tokens reserved for reasoning.
- `max_tokens` must exceed the thinking budget. With a 1024-token budget you need `max_tokens ≥ 1025`.
- You're charged for thinking tokens, and they add latency.

### Response shape
| Block type | Contains |
|---|---|
| `thinking` | Reasoning text + cryptographic `signature` (prevents tampering with thinking text on round-trip) |
| `text` | The final user-facing response |
| `redacted_thinking` | Encrypted reasoning that Claude's safety systems flagged. Still passed back so multi-turn context stays intact. |

### When to use
Reach for extended thinking **after** prompt optimization fails to hit the accuracy you need. Use prompt evals to decide whether the cost/latency trade-off is worth it — don't enable it by default.

### Forcing redacted thinking
Anthropic publishes a magic test string that deliberately triggers `redacted_thinking` so you can exercise that code path locally:
```
ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB
```

### Demo: thinking on, then thinking off

Same word problem, two requests — once with `thinking=True` and once without — and we print the **block types** that came back, plus a snippet of the reasoning if any. The contrast students should notice: with thinking on, the response object contains a `thinking` block before the final `text` block; without it, only a `text` block.

Swap the question for something genuinely arithmetic-heavy (e.g., *"Multiply 47 by 83 and verify by long division"*) to see the reasoning block stretch closer to the budget.

In [ ]:
question = "I have 27 apples. I give 1/3 to Alice, then double what's left. How many apples now?"

# --- WITH extended thinking ---
msgs = [{"role": "user", "content": question}]
resp_thinking = chat(msgs, thinking=True, thinking_budget=1024, max_tokens=2000)

print("=== thinking=True ===")
print("Block types:", [b.type for b in resp_thinking.content])
for b in resp_thinking.content:
    if b.type == "thinking":
        print("\n--- Reasoning (first 300 chars) ---")
        print(b.thinking[:300] + ("..." if len(b.thinking) > 300 else ""))
print("\nFinal answer:", text_from_message(resp_thinking))

# --- WITHOUT extended thinking ---
msgs = [{"role": "user", "content": question}]
resp_plain = chat(msgs)

print("\n=== thinking=False ===")
print("Block types:", [b.type for b in resp_plain.content])
print("Answer:", text_from_message(resp_plain))

> **🏫 During class:**
> 1. Run the cell. Point at the **Block types** line in each output: thinking-on returns `['thinking', 'text']`, thinking-off returns `['text']`.
> 2. Say out loud: *"This is the model showing its work — and you're paying for those tokens. Use it when accuracy matters more than cost."*
> 3. Variation: paste the magic string from the section above into a new user message with `thinking=True` and watch the block type flip to `redacted_thinking`. Ask: *"Why would Anthropic redact the reasoning but still send the block?"* — answer: so multi-turn conversations don't lose context.

---
# 2. Image Support

Claude can analyze images directly inside a user message — counting, comparing, describing, classifying. Images are content blocks, not a separate API.

### Limits
- **Up to 100 images per request.**
- Size and dimension caps apply.
- Images are tokenized based on pixel area; large images cost more input tokens.

### Block shape
An image block lives inside the user message's `content` list, alongside text:

```python
{"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": "..."}}
# or
{"type": "image", "source": {"type": "url", "url": "https://..."}}
```

You can mix several image blocks and text blocks in the same message.

### The catch
**Image accuracy depends almost entirely on prompt sophistication.** A bare *"what's in this image?"* yields shallow output. Real production prompts use:

- Step-by-step analysis instructions
- Few-shot examples (alternating image/text pairs)
- Explicit verification steps
- Numerical scoring rubrics

A reference example from the lecture notes: automated **fire risk assessment** from satellite imagery — analyze tree density, roof overhang, defensible space, then assign a 1–4 risk rating with justification. The image hasn't changed; the rubric is what makes the answer reliable.

### Demo: lazy prompt vs. structured prompt

Same image (a public-domain cat photo from Wikipedia, fetched by URL), two prompts:
1. A bare *"describe this image"* — quick, generic output.
2. A structured prompt that asks for specific observations + a confidence assessment.

Watch how much sharper the structured prompt's output is. The image hasn't changed; only the instructions have.

In [ ]:
image_url = "https://upload.wikimedia.org/wikipedia/commons/3/3a/Cat03.jpg"

# --- A. Bare prompt ---
msgs = [
    {"role": "user", "content": [
        {"type": "image", "source": {"type": "url", "url": image_url}},
        {"type": "text", "text": "Describe this image."},
    ]}
]
print("=== Lazy prompt ===")
print(text_from_message(chat(msgs, max_tokens=400)))

# --- B. Structured prompt ---
structured = """
Analyze the attached image with these specific steps:
1. Subject: identify the primary subject and any secondary subjects.
2. Setting: describe the environment, lighting, and time of day.
3. Composition: note framing, camera angle, and visual focal point.
4. Notable details: 3 specific things a casual viewer might miss.
5. Confidence: rate your overall confidence (low / medium / high) and say why.

Answer each step on its own line, prefixed with the step number.
"""

msgs = [
    {"role": "user", "content": [
        {"type": "image", "source": {"type": "url", "url": image_url}},
        {"type": "text", "text": structured},
    ]}
]
print("\n=== Structured prompt ===")
print(text_from_message(chat(msgs, max_tokens=600)))

> **🏫 During class:**
> 1. Run the cell. Read the lazy answer aloud — *"that's adequate, not great."*
> 2. Read the structured answer. Point at the difference: *"Same image, same model. The only thing that changed is the prompt structure."*
> 3. Variation: switch the URL to a satellite-imagery URL and adapt the structured prompt to the fire-risk rubric from the notes. The point lands harder when students see real numerical scoring.
> 4. Question for the room: *"If you wanted to OCR a receipt for line items, would you use a bare prompt?"* — answer: no, you'd give it explicit field-extraction instructions and probably a JSON schema example.

---
# 3. PDF Support

PDFs work almost identically to images — **swap the block type and the media type**, keep the rest of your code:

| | Image | PDF |
|---|---|---|
| `type` | `"image"` | `"document"` |
| `media_type` | `"image/png"` | `"application/pdf"` |
| Reads | pixels | text + images + charts + tables |

That last row is the headline: Claude reads **everything** in the PDF — paragraph text, tables, embedded charts, figure captions — in a single call. No OCR pipeline, no separate table extractor, no PDF-to-text preprocessor.

### Use cases
- Extracting numbers from financial reports
- Summarizing legal contracts
- Pulling structured data out of scanned forms
- Citing specific pages back (see §4)

### Demo: ask a question against earth.pdf

We base64-encode `features_of_claude/earth.pdf` (the Wikipedia "Earth" article saved as a PDF) and ask a question that requires reading text **plus** the data tables in the sidebar. Claude returns a single answer fusing both — no preprocessing on our side.

Try changing the question to something that lives only in a chart or a sidebar (e.g., *"What is Earth's mean orbital speed?"*) and confirm the PDF extraction handles it.

In [ ]:
import base64

with open("features_of_claude/earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

msgs = [
    {"role": "user", "content": [
        {"type": "document", "source": {
            "type": "base64",
            "media_type": "application/pdf",
            "data": file_bytes,
        }},
        {"type": "text", "text":
            "What is Earth's mean orbital speed, and what is its axial tilt? "
            "Cite the page if you can."},
    ]}
]

resp = chat(msgs, max_tokens=600)
print(text_from_message(resp))

> **🏫 During class:**
> 1. Before running, ask: *"What would this take in a non-LLM stack?"* — likely answer: pdfplumber for text + a table parser + glue code.
> 2. Run the cell. Point out the model picks up numbers from the data table on the side of the article, not just paragraph text.
> 3. Variation: ask a question that requires *summarizing across* paragraphs (e.g., *"Summarize how Earth's atmosphere formed in 3 sentences."*). Confirm the model is reading prose, not just keyword-searching.
> 4. Setup hint to share: *"The encoding pattern is identical to images. Once you've done one, you've done both."*

---
# 4. Citations

**Citations** make Claude tell you where in a source document each part of its answer came from. The model returns text blocks, and any block grounded in the source carries a `citations` array describing the exact location.

### Two citation types
| Type | Source | Location data |
|---|---|---|
| `citation_page_location` | PDF document | document index, document title, **start page**, **end page**, cited text |
| `citation_char_location` | Plain text block | document index, document title, **start char index**, **end char index**, cited text |

### Enabling citations
Two flags on the source block:
- `"citations": {"enabled": True}`
- `"title": "..."` — a label that flows into every citation back-reference, so your UI can name the source.

Works for both PDF documents and `text/plain` documents.

### Why use them
- **Trust:** users can verify a quoted figure or claim is real, not hallucinated.
- **UX:** popovers and footnotes — hover an answer fragment, see the page/section it came from.
- **Audit trail:** logging the citations alongside the answer gives you a record of what the model actually grounded on.

### Demo: ask a question, inspect the citations array

We pass a plain-text article (no PDF this time) with `citations.enabled = true` and a `title`. Claude's response is a list of text blocks; some carry a `citations` array, some don't. We print each block's text + its citation metadata so students can see the structure.

Switch `enabled` to `False` and re-run to see the same answer come back without any citation arrays — visually identical text, but no source data attached.

In [ ]:
article = """
The Apollo 11 mission landed humans on the Moon on July 20, 1969.
The mission was crewed by Neil Armstrong, Buzz Aldrin, and Michael Collins.
Armstrong became the first person to step onto the lunar surface, followed by Aldrin
about 19 minutes later. Collins remained in lunar orbit aboard the Command Module Columbia.
The crew returned safely to Earth on July 24, 1969, splashing down in the Pacific Ocean.
"""

msgs = [{"role": "user", "content": [
    {
        "type": "document",
        "source": {"type": "text", "media_type": "text/plain", "data": article},
        "title": "Apollo 11 brief",
        "citations": {"enabled": True},
    },
    {"type": "text", "text": "Who stepped on the Moon first, and when did the crew return to Earth?"},
]}]

resp = chat(msgs, max_tokens=600)

for i, block in enumerate(resp.content):
    if block.type != "text":
        continue
    print(f"--- block {i} ---")
    print("TEXT:", block.text)
    cits = getattr(block, "citations", None) or []
    for c in cits:
        print("  CITATION:")
        print("    title         :", c.document_title)
        print("    cited_text    :", c.cited_text)
        print("    char range    :", (c.start_char_index, c.end_char_index))
    print()

> **🏫 During class:**
> 1. Run the cell. Walk through the printed blocks: most are plain text, a couple carry a `citations` array with the **exact substring** Claude grounded on.
> 2. Say out loud: *"This is what powers the hover-popover UX you see in Claude.ai. The model gives you everything you need to render footnotes."*
> 3. Variation: re-run with `enabled: False` (or comment that line out) and show that the answer still works — but the citation array is gone. *"Citations are extra metadata. The answer text is unchanged."*
> 4. Question: *"What changes if I swap the plain-text source for a PDF?"* — answer: citations carry **page numbers** (`start_page_number` / `end_page_number`) instead of char indices.

---
# 5. Prompt Caching — the concept

**Prompt Caching** stores the computational work Claude does on your input messages so subsequent requests with identical content can skip that work. You pay less, and the response starts faster.

### Default flow (no cache)
```
User → API → Claude processes input (build internal data structures) → generate output → discard processing → ready for next request
```
Every follow-up that contains the same long system prompt or tool schema makes Claude redo all of that work from scratch.

### With cache
```
User → API → process input → SAVE processed work to cache → generate output
... follow-up with identical prefix ...
User → API → READ cached work → generate output
```
The redundant work is reused. The longer the cached prefix, the bigger the win.

### Where it shines
- Long, identical system prompts across thousands of users
- Static tool-schema bundles
- Repeated message prefixes (knowledge base, chat conversations with shared context)

The next two sections cover the **rules** for triggering caching and a **live demo** of the savings.

### Demo: the same call twice — no breakpoint, no savings

Before we even add a cache breakpoint, let's prove the **default** behavior: identical follow-up calls don't reuse work. We run the same long-system-prompt request twice and print the cache token counts on each `usage` object. Both should show `cache_creation_input_tokens=0` and `cache_read_input_tokens=0` — i.e., nothing was cached.

This sets up §6 (*"we have to opt in"*) and §7 (*"here's the savings"*).

In [ ]:
LONG_SYSTEM = (
    "You are a senior compliance analyst at a mid-size bank. "
    "Answer in three short paragraphs covering: (1) regulatory framing, "
    "(2) operational controls, (3) audit-trail expectations. "
    "Use precise banking-regulator language and never speculate beyond the cited rule. "
) * 8  # repeat to comfortably exceed the 1024-token caching threshold

msgs = [{"role": "user", "content": "What's the difference between SOX and SOC 2?"}]

print("=== Call #1 (no caching) ===")
r1 = chat(msgs, system=LONG_SYSTEM, max_tokens=400)
print("input_tokens                :", r1.usage.input_tokens)
print("cache_creation_input_tokens :", r1.usage.cache_creation_input_tokens)
print("cache_read_input_tokens     :", r1.usage.cache_read_input_tokens)

print("\n=== Call #2 (no caching, identical input) ===")
r2 = chat(msgs, system=LONG_SYSTEM, max_tokens=400)
print("input_tokens                :", r2.usage.input_tokens)
print("cache_creation_input_tokens :", r2.usage.cache_creation_input_tokens)
print("cache_read_input_tokens     :", r2.usage.cache_read_input_tokens)

> **🏫 During class:**
> 1. Run the cell. Point at the four cache-related token counts: all zeros on both calls.
> 2. Key sentence to say out loud: *"Even though the second request was byte-identical, Claude reprocessed the entire system prompt. Caching is not the default — you opt in."*
> 3. Foreshadow §7: *"Hold this `input_tokens` number in mind — it's what we'll watch shrink once we add a cache breakpoint."*

---
# 6. Rules of Prompt Caching

Caching has a small set of rules. Memorize these and most production caching bugs disappear.

### The mechanism
1. Initial request → Claude processes the input → **saves** to cache → returns response.
2. Follow-up with identical prefix → Claude **reads** the cache → skips reprocessing → returns response.

### Hard rules
| Rule | Detail |
|---|---|
| Cache duration | **1 hour** maximum. Idle entries get evicted. |
| Activation | Manual — you must add a `cache_control` breakpoint to a content block. |
| Minimum cache size | **1024 tokens**. Smaller prefixes won't be cached. |
| Max breakpoints per request | **4** |
| Order of caching | tools → system prompt → messages (concatenated in that order) |
| Cache scope | All content **up to and including** the breakpoint |
| Invalidation | Any change to content **before** the breakpoint busts the entire cache |

### Where you can place breakpoints
- Tool schemas (typically the last tool in the list)
- System prompt
- Any message block — text, image, tool_use, tool_result

### Shorthand vs. longhand
You can only add `cache_control` to a content block in **longhand** form:

```python
# Shorthand (no caching possible)
content = "some text"

# Longhand (caching possible)
content = [{"type": "text", "text": "some text", "cache_control": {"type": "ephemeral"}}]
```

### Multiple breakpoints
With up to 4 breakpoints you can build **layered** caches:
- Breakpoint A on the tool schemas (rarely changes).
- Breakpoint B on the system prompt (sometimes changes).
- Breakpoint C on a static knowledge-base prefix in the first user message.

If only the very last user message changes, all three layers stay warm. If the system prompt changes, A still hits but B and C are invalidated.

### Demo: adding a single breakpoint to the system prompt

Same long system prompt as §5, but this time we wrap it in the longhand form with a `cache_control` breakpoint. We run the request twice. The first call **writes** to the cache (`cache_creation_input_tokens > 0`); the second call **reads** from the cache (`cache_read_input_tokens > 0`). Cached tokens are billed at a much lower rate than fresh input.

If you change a single character anywhere inside `LONG_SYSTEM` and re-run, both counters reset on the next call — that's invalidation in action.

In [ ]:
# Longhand system prompt with a cache breakpoint
system_blocks = [
    {
        "type": "text",
        "text": LONG_SYSTEM,
        "cache_control": {"type": "ephemeral"},
    }
]

msgs = [{"role": "user", "content": "What's the difference between SOX and SOC 2?"}]

print("=== Call #1 (writes cache) ===")
r1 = chat(msgs, system=system_blocks, max_tokens=400)
print("input_tokens                :", r1.usage.input_tokens)
print("cache_creation_input_tokens :", r1.usage.cache_creation_input_tokens)
print("cache_read_input_tokens     :", r1.usage.cache_read_input_tokens)

print("\n=== Call #2 (reads cache) ===")
r2 = chat(msgs, system=system_blocks, max_tokens=400)
print("input_tokens                :", r2.usage.input_tokens)
print("cache_creation_input_tokens :", r2.usage.cache_creation_input_tokens)
print("cache_read_input_tokens     :", r2.usage.cache_read_input_tokens)

> **🏫 During class:**
> 1. Run the cell. Read the four token counts on each call out loud:
>    - Call #1 → `cache_creation_input_tokens` is large, `cache_read_input_tokens = 0` ⇒ paid the full input price plus a small write surcharge.
>    - Call #2 → `cache_creation_input_tokens = 0`, `cache_read_input_tokens` is large ⇒ paid only the (cheaper) cache-read rate.
> 2. *"That's the savings. Long static prefix + breakpoint = pay once, read for an hour."*
> 3. Variation: append a single space to the end of `LONG_SYSTEM` (so it's a one-character difference) and re-run. The next call should look like a brand-new cache — invalidation in action.
> 4. Question: *"If my system prompt is 800 tokens, will caching help?"* — answer: no, you're below the 1024-token minimum.

---
# 7. Prompt Caching in Action

A practical pattern: bake caching into your `chat()` helper so **tool schemas** and **system prompts** are cached automatically across the whole app. Most production codebases do this once and forget about it.

### What we'll do
- Cache the **last tool schema** (a breakpoint on the tail of the tool list caches the whole list as a layer).
- Cache the **system prompt** (wrapped in a longhand text block).
- Use **two breakpoints** in one request — both layers can hit independently.

### Best practice when modifying tools
**Don't mutate the original tool list.** A cached helper that mutates its input causes mysterious bugs the first time the same tool list is reused on a non-cached call. Make a copy, replace the last tool with a clone that has `cache_control`, then send the copy.

### Reading the metering
| Field | Meaning |
|---|---|
| `cache_creation_input_tokens` | Tokens written to cache on this call |
| `cache_read_input_tokens` | Tokens served from cache on this call |
| `input_tokens` | Tokens billed at the standard input rate (un-cached) |

Partial cache hits are common: with two breakpoints, you might read tools from cache (hit) but write a fresh system-prompt cache (miss) on the same call.

### Invalidation reminder
Anything before the breakpoint changes → cache busts. Identical prefix → cache hits.

### Demo: a caching-enabled chat helper

We define a small `cached_chat()` wrapper that:
- Clones the last tool with `cache_control` attached.
- Wraps the system prompt in a longhand block with `cache_control`.
- Sends the request and returns the full response.

Then we run two calls with the same tools + system prompt. The first call writes both layers; the second reads both. If you change one tool's description and re-run, you'll see partial hits — system prompt still cached, tool list re-cached.

In [ ]:
import copy

def cached_chat(messages, system, tools, max_tokens=400):
    # Cache the last tool schema (caches the whole tool list as a layer).
    tools_cached = list(tools)
    last = copy.deepcopy(tools_cached[-1])
    last["cache_control"] = {"type": "ephemeral"}
    tools_cached[-1] = last

    # Cache the system prompt as the second layer.
    system_blocks = [{
        "type": "text",
        "text": system,
        "cache_control": {"type": "ephemeral"},
    }]

    return client.messages.create(
        model=model,
        max_tokens=max_tokens,
        system=system_blocks,
        tools=tools_cached,
        messages=messages,
    )

# Reuse the long system prompt from §5 + a small tool schema set.
TOOLS = [
    {
        "name": "get_regulation_text",
        "description": (
            "Return the full text of a named regulation. "
            "Pass the regulation ID (e.g., 'SOX-302' or 'SOC2-CC1.2'). "
            "Use this tool any time you need to ground an answer in the exact "
            "regulatory language rather than paraphrase."
        ),
        "input_schema": {
            "type": "object",
            "properties": {"regulation_id": {"type": "string"}},
            "required": ["regulation_id"],
        },
    },
    {
        "name": "list_audit_findings",
        "description": (
            "List the most recent N audit findings for a given control area. "
            "Use to inform compliance answers with current evidence rather than "
            "guessing. Returns finding ID, severity, and short summary for each."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "control_area": {"type": "string"},
                "limit": {"type": "integer", "default": 10},
            },
            "required": ["control_area"],
        },
    },
]

msgs = [{"role": "user", "content": "What's the difference between SOX and SOC 2?"}]

print("=== Call #1 (writes both cache layers) ===")
r1 = cached_chat(msgs, system=LONG_SYSTEM, tools=TOOLS)
print("input_tokens                :", r1.usage.input_tokens)
print("cache_creation_input_tokens :", r1.usage.cache_creation_input_tokens)
print("cache_read_input_tokens     :", r1.usage.cache_read_input_tokens)

print("\n=== Call #2 (reads both cache layers) ===")
r2 = cached_chat(msgs, system=LONG_SYSTEM, tools=TOOLS)
print("input_tokens                :", r2.usage.input_tokens)
print("cache_creation_input_tokens :", r2.usage.cache_creation_input_tokens)
print("cache_read_input_tokens     :", r2.usage.cache_read_input_tokens)

> **🏫 During class:**
> 1. Run the cell. Walk the room through the four numbers per call. Highlight that **call #2's `cache_read_input_tokens`** is now larger than in §6 — both the tools and the system prompt got cached.
> 2. Point at the `copy.deepcopy(tools_cached[-1])` line. Say: *"This matters. If we mutated the original tool list, every other code path that uses that list would also send `cache_control`. That's the bug we're avoiding."*
> 3. Variation: edit the `description` of `list_audit_findings` and re-run. You should see `cache_creation_input_tokens` increase (tool layer re-cached) while system-prompt tokens still come from cache — that's a **partial cache hit**.
> 4. Question: *"How many breakpoints do I have left?"* — answer: 2. We've used 2 of the maximum 4. You could add another for a static knowledge base in the user message, and another for a long conversation prefix.

---
# 8. Code Execution and the Files API

Two server-side features that pair naturally:

### Files API
Upload a file once, get back a stable `file_id`, reference that ID in any future request. No more re-encoding the same CSV every call.

```
upload(file_path) → File metadata { id, filename, mime_type, ... }
... later ...
{"type": "container_upload", "file_id": "file_..."}  # passed in a user message
```

### Code Execution
A built-in tool — you do **not** implement it. Just include the predefined tool schema:

```python
tools = [{"type": "code_execution_20250825", "name": "code_execution"}]
```

Claude runs Python in an isolated **Docker container** with **no network access**. It can write code, run it, read the output, fix bugs, and iterate — all inside one response.

### Combined flow
```
1. Upload a CSV via Files API → get file_id
2. Send a user message with:
   • container_upload block referencing file_id
   • a text block describing the analysis you want
3. Pass the code_execution tool
4. Claude writes Python → executes → interprets results → may write more code → returns final analysis
5. Any files Claude generated (plots, exports) come back as new file_ids you can download
```

### Why this combo matters
Without it, *"do data analysis on this CSV"* means **you** write the pandas code. With it, you describe the question and Claude does the analysis end-to-end — including plots, summary stats, and follow-up checks.

### Constraints
- **No network** in the container.
- Each code execution starts with a **clean kernel** — variables and imports do not persist across executions inside one response.

### Demo: upload a CSV, ask for analysis, inspect the response shape

We upload `features_of_claude/streaming.csv` via the Files API, send Claude a user message that combines a `container_upload` block (referencing the file_id) with an analysis prompt, and pass the `code_execution` tool. Claude writes pandas code, runs it, interprets the result, and (often) generates a chart that comes back as a new file_id we could download.

The key thing to notice is the **content blocks** in the response: a mix of `text` (Claude narrating its analysis), `server_tool_use` (the code it ran), and `code_execution_tool_result` (what the container printed back). It's a one-shot data-science workflow.

In [ ]:
from pathlib import Path

# --- File helpers (Files API) ---
def upload_file(path):
    p = Path(path)
    mime = {
        ".csv": "text/csv",
        ".pdf": "application/pdf",
        ".txt": "text/plain",
        ".png": "image/png",
        ".jpg": "image/jpeg",
        ".xlsx": "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
    }[p.suffix.lower()]
    with open(p, "rb") as fh:
        return client.beta.files.upload(file=(p.name, fh, mime))

def download_file(file_id, dest):
    client.beta.files.download(file_id).write_to_file(dest)

# Upload the CSV
file_meta = upload_file("features_of_claude/streaming.csv")
print("Uploaded:", file_meta.id, file_meta.filename)

# Ask Claude to analyze it
msgs = [{"role": "user", "content": [
    {"type": "text", "text":
        "Analyze the attached CSV. Identify the top 2 drivers of churn and "
        "produce a labeled matplotlib chart that supports your conclusion. "
        "Note: every code execution starts with a clean Python kernel — "
        "redeclare any imports/variables you need each time."
    },
    {"type": "container_upload", "file_id": file_meta.id},
]}]

resp = chat(
    msgs,
    tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
    max_tokens=8000,
)

# Walk the response and surface the structure
print("\n--- Response block types ---")
for b in resp.content:
    print(" ", b.type)

print("\n--- Final narration ---")
print(text_from_message(resp))

> **🏫 During class:**
> 1. Run the upload + analysis cell. While it runs, narrate the wait: *"Claude is writing Python, executing it inside a Docker container, reading the result, and deciding what to do next — possibly running more code."*
> 2. When the cell finishes, walk through the **block-types list**. Point at every `server_tool_use` (the code) and `code_execution_tool_result` (what came back). *"This is the agentic loop, and you didn't write any of it."*
> 3. If the response includes a generated chart `file_id`, run a follow-up cell live: `download_file("<file_id>", "churn.png")` and open it.
> 4. Question for the room: *"What constraints does the no-network sandbox impose on what kinds of analyses you can ship?"* — discussion: anything pulling from external APIs has to be staged via the Files API; static data analysis works great.

---
# 9. Recap + practice exercises

### Recap
- **Extended Thinking** — opt-in reasoning budget; charged + slower; reach for it after prompt eval shows you need it.
- **Image Support** — same content-block shape as text; **prompt sophistication** is what makes accuracy land.
- **PDF Support** — change `image` → `document`, `image/png` → `application/pdf`; everything else is the same.
- **Citations** — flip `enabled: True`, set a `title`, parse the `citations` array on each text block.
- **Prompt Caching** — opt-in via `cache_control: {type: "ephemeral"}`; 1h TTL; 1024-token min; max 4 breakpoints; order is tools → system → messages.
- **Caching in production** — bake into a wrapper, clone (don't mutate) the last tool, watch `cache_creation_input_tokens` and `cache_read_input_tokens`.
- **Files API + Code Execution** — upload once, reference by ID; Claude runs Python in an isolated container, end-to-end data work in one call.

### Exercises
1. **Thinking eval:** Run the same hard math question 5 times with `thinking=False` and 5 times with `thinking=True, thinking_budget=4000`. Tally accuracy and median latency. Decide whether thinking is worth it for that prompt class.
2. **Image rubric:** Build a prompt that scores property photos on a 1–5 "curb appeal" scale with explicit criteria. Pass 3 images in one request and have Claude score them comparatively.
3. **PDF Q&A with citations:** Use `earth.pdf` with citations enabled. Ask 3 different questions and render the citations as `[page X]` footnote markers in the printed answer.
4. **Cache savings calculator:** Wrap `cached_chat` with a counter that prints estimated dollars saved (cache reads vs. uncached input tokens) over N calls. Run a 20-call simulated chat session.
5. **Files-API analyst:** Upload a different CSV (your own data), ask Claude to find a non-obvious correlation, and download the resulting plot. Then ask a follow-up question that requires re-reading the file — confirm the `file_id` still works on the second turn.

In [ ]:
# Exercise 1 scaffold — finish this live in class together.
# import time, statistics
#
# question = (
#     "Three cars travel 60, 75, and 90 km/h. They start at the same point and drive for 2.5h. "
#     "Then car 1 turns back, car 2 stops, car 3 continues for 1h more. Where is each at t=3.5h?"
# )
#
# def run(thinking):
#     t0 = time.time()
#     r = chat([{"role": "user", "content": question}],
#              thinking=thinking, thinking_budget=4000, max_tokens=6000)
#     return time.time() - t0, text_from_message(r)
#
# results = {"on": [], "off": []}
# for _ in range(5):
#     results["off"].append(run(False))
#     results["on"].append(run(True))
#
# for label, runs in results.items():
#     latencies = [r[0] for r in runs]
#     print(label, "median latency:", round(statistics.median(latencies), 2), "s")